In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ==========================================
# 1. Activation Functions & Utilities
# ==========================================
def sigmoid(Z):
    # Clip Z to avoid overflow warnings in exp
    Z = np.clip(Z, -500, 500)
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_backward(Z):
    return (Z > 0).astype(float)

# ==========================================
# 2. Neural Network Core Functions
# ==========================================
def init_parameters(n_x, n_h, n_y):
    """
    n_x: size of input layer
    n_h: size of hidden layer
    n_y: size of output layer
    """
    np.random.seed(42)
    # He initialization for ReLU
    W1 = np.random.randn(n_h, n_x) * np.sqrt(2. / n_x)
    b1 = np.zeros((n_h, 1))
    # Xavier initialization for Sigmoid
    W2 = np.random.randn(n_y, n_h) * np.sqrt(1. / n_h)
    b2 = np.zeros((n_y, 1))

    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

def forward_pass(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def cost_function(A2, Y):
    m = Y.shape[1]
    # Add epsilon to prevent log(0)
    epsilon = 1e-15
    cost = - (1/m) * np.sum(Y * np.log(A2 + epsilon) + (1 - Y) * np.log(1 - A2 + epsilon))
    return np.squeeze(cost)

def backward_pass(parameters, cache, X, Y):
    m = X.shape[1]
    W2 = parameters["W2"]
    A1, A2, Z1 = cache["A1"], cache["A2"], cache["Z1"]

    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_backward(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

def fit(X, Y, layers_dims, learning_rate=0.01, num_epochs=1000, print_cost=True):
    n_x, n_h, n_y = layers_dims
    parameters = init_parameters(n_x, n_h, n_y)

    costs = []

    for i in range(num_epochs):
        A2, cache = forward_pass(X, parameters)
        cost = cost_function(A2, Y)
        grads = backward_pass(parameters, cache, X, Y)

        # Update parameters
        parameters["W1"] -= learning_rate * grads["dW1"]
        parameters["b1"] -= learning_rate * grads["db1"]
        parameters["W2"] -= learning_rate * grads["dW2"]
        parameters["b2"] -= learning_rate * grads["db2"]

        if print_cost and i % 100 == 0:
            print(f"Cost after epoch {i}: {cost:.4f}")
            costs.append(cost)

    return parameters

def predict(X, parameters):
    A2, _ = forward_pass(X, parameters)
    return (A2 > 0.5).astype(int)

# ==========================================
# 3. Data Preprocessing & Execution
# ==========================================
def load_and_preprocess_data():
    print("Loading data...")
    train_df = pd.read_csv('/customer_churn_dataset-training-master.csv')
    test_df = pd.read_csv('/customer_churn_dataset-testing-master.csv')

    # Drop NAs
    train_df.dropna(inplace=True)
    test_df.dropna(inplace=True)

    # Separate features and labels
    X_train_raw = train_df.drop(['CustomerID', 'Churn'], axis=1)
    Y_train_raw = train_df['Churn'].values.reshape(1, -1)

    X_test_raw = test_df.drop(['CustomerID', 'Churn'], axis=1)
    Y_test_raw = test_df['Churn'].values.reshape(1, -1)

    # Handle Categorical variables (Gender, Subscription Type, Contract Length)
    # We combine them temporarily to ensure consistent encoding
    combined = pd.concat([X_train_raw, X_test_raw])
    combined = pd.get_dummies(combined, drop_first=True)

    # Split back
    X_train = combined.iloc[:len(train_df)]
    X_test = combined.iloc[len(train_df):]

    # Standardize numerical features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).T # Transpose for NN shape (features, samples)
    X_test_scaled = scaler.transform(X_test).T

    return X_train_scaled, Y_train_raw, X_test_scaled, Y_test_raw

if __name__ == "__main__":
    X_train, Y_train, X_test, Y_test = load_and_preprocess_data()

    print(f"\nData shapes:\nX_train: {X_train.shape}, Y_train: {Y_train.shape}")
    print(f"X_test: {X_test.shape}, Y_test: {Y_test.shape}\n")

    n_x = X_train.shape[0] # Number of features
    n_y = 1 # Binary classification

    # Testing different hidden layer sizes as requested by the assignment
    hidden_layer_sizes = [4, 8, 16]
    learning_rate = 0.05
    epochs = 1000

    for n_h in hidden_layer_sizes:
        print(f"--- Training Model with Hidden Layer Size: {n_h} ---")
        layers_dims = (n_x, n_h, n_y)

        trained_parameters = fit(X_train, Y_train, layers_dims, learning_rate, epochs, print_cost=False)

        # Calculate Accuracies
        train_predictions = predict(X_train, trained_parameters)
        test_predictions = predict(X_test, trained_parameters)

        train_accuracy = np.mean(train_predictions == Y_train) * 100
        test_accuracy = np.mean(test_predictions == Y_test) * 100

        print(f"Train Accuracy: {train_accuracy:.2f}%")
        print(f"Test Accuracy: {test_accuracy:.2f}%\n")

Loading data...

Data shapes:
X_train: (12, 440832), Y_train: (1, 440832)
X_test: (12, 64374), Y_test: (1, 64374)

--- Training Model with Hidden Layer Size: 4 ---
Train Accuracy: 90.23%
Test Accuracy: 57.16%

--- Training Model with Hidden Layer Size: 8 ---
Train Accuracy: 90.12%
Test Accuracy: 56.98%

--- Training Model with Hidden Layer Size: 16 ---
Train Accuracy: 92.05%
Test Accuracy: 55.81%

